In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn transformers torch tqdm peft')
    print("Setup complete!")


In [2]:
# NOTE: If running this notebook manually in the IDE, make sure you have transformers and torch installed!
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_finetuned_results'

MODELS_TO_TEST = [
    'models/finetuned/xlmr/xlmr_no_rehearsal',
    'models/finetuned/xlmr/xlmr_with_rehearsal'
]


In [3]:
import os
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "en", "tam": "ta", "hin": "hi", "ben": "bn", "arb": "ar", "fra": "fr", "deu": "de",
    "jpn": "ja", "nld": "nl", "pol": "pl", "ita": "it", "por": "pt", "tur": "tr", "spa": "es", "ell": "el", "urd": "ur", "bul": "bg", "cmn": "zh", "rus": "ru", "tha": "th", "swh": "sw", "vie": "vi",
    "sinhala": "si", "sanskrit": "sa", "pali": "pi",
    "san": "sa_sinh"  # Separate target for Sinhala-scripted Sanskrit evaluation
}

def load_dataset(file_path):
    print(f"\nLoading {os.path.basename(file_path)}...")
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label") in TARGET_LANGUAGES:
                records.append(row)
    df = pd.DataFrame(records)
    if not df.empty:
        df["mapped_label"] = df["label"].map(TARGET_LANGUAGES)
        print(f"Loaded {len(df)} rows across {df['label'].nunique()} target languages")
    else:
        print("No matching target languages found in this dataset.")
    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels, zero_division=0
    )

    print("\n" + "=" * 48)
    print(f"FINETUNED BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4, zero_division=0
    ))

    os.makedirs(output_dir, exist_ok=True)
    safe_model_name = os.path.basename(model_name).replace('/', '_').replace(' ', '_').replace('-', '_').lower()
    out_file = os.path.join(output_dir, f"{safe_model_name}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*_integrated.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [4]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from tqdm.auto import tqdm
import torch
import torch.distributed.tensor  # Fix peft bug
from peft import PeftModel

device = 0 if torch.cuda.is_available() else -1
target_labels = sorted(set(TARGET_LANGUAGES.values()))

for model_path in MODELS_TO_TEST:
    print(f"\n{'#'*60}")
    print(f"BENCHMARKING MODEL: {model_path}")
    print(f"{'#'*60}\n")
    
    if not os.path.exists(model_path):
        print(f"[WARNING] Model path {model_path} does not exist. Skipping... (Have you run the finetuning notebook yet?)")
        continue

    config = AutoConfig.from_pretrained(model_path)
    base_model = AutoModelForSequenceClassification.from_pretrained(config._name_or_path, config=config, ignore_mismatched_sizes=True)
    model = PeftModel.from_pretrained(base_model, model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)
    
    for file_path in dataset_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df = load_dataset(file_path)
        if df.empty: continue
        
        texts = df["text"].astype(str).tolist()
        print(f"Evaluating {len(texts)} samples with {os.path.basename(model_path)}...")
        
        preds = pipe(texts, batch_size=32, truncation=True, max_length=512)
        raw_predicted_labels = [p['label'] for p in preds]

        results = df[["text", "label", "source"]].copy()
        results["true_label"] = df["mapped_label"]
        
        # XLM-R uses a single 'sa' node for both Sanskrit scripts. 
        # To evaluate them separately, if the model predicts 'sa' for a 'sa_sinh' true label, we consider it correct for 'sa_sinh'.
        final_preds = []
        for true_l, pred_l in zip(results["true_label"], raw_predicted_labels):
            if pred_l == "sa" and true_l == "sa_sinh":
                final_preds.append("sa_sinh")
            else:
                final_preds.append(pred_l)
                
        results["predicted_label"] = final_preds

        evaluate_and_save(results, os.path.basename(model_path), dataset_name, target_labels)


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



############################################################
BENCHMARKING MODEL: models/finetuned/xlmr/xlmr_no_rehearsal
############################################################



Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at papluca/xlm-roberta-base-language-detection and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([20]) in the checkpoint and torch.Size([25]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([20, 768]) in the checkpoint and torch.Size([25, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
The model 'PeftModelForSequenceC


Loading wili-2018_integrated.jsonl...
Loaded 27374 rows across 23 target languages
Evaluating 27374 samples with xlmr_no_rehearsal...

FINETUNED BENCHMARK RESULTS (xlmr_no_rehearsal on wili-2018_integrated)
Accuracy:  30.50%
Macro F1:  12.34%

Per-language breakdown:

              precision    recall  f1-score   support

          ar     0.0000    0.0000    0.0000         0
          bg     0.0000    0.0000    0.0000      1000
          bn     0.0000    0.0000    0.0000      1000
          de     0.0000    0.0000    0.0000      1000
          el     0.0000    0.0000    0.0000      1000
          en     0.0000    0.0000    0.0000      1000
          es     0.0000    0.0000    0.0000      1000
          fr     0.0000    0.0000    0.0000      1000
          hi     0.0000    0.0000    0.0000      1000
          it     0.0000    0.0000    0.0000      1000
          ja     0.0000    0.0000    0.0000      1000
          nl     0.0000    0.0000    0.0000      1000
          pi     0.9921    

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at papluca/xlm-roberta-base-language-detection and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([20]) in the checkpoint and torch.Size([25]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([20, 768]) in the checkpoint and torch.Size([25, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
[W924 10:03:47.886373825 CUDACac


Loading wili-2018_integrated.jsonl...
Loaded 27374 rows across 23 target languages
Evaluating 27374 samples with xlmr_with_rehearsal...

FINETUNED BENCHMARK RESULTS (xlmr_with_rehearsal on wili-2018_integrated)
Accuracy:  94.67%
Macro F1:  82.47%

Per-language breakdown:

              precision    recall  f1-score   support

          ar     0.0000    0.0000    0.0000         0
          bg     0.0000    0.0000    0.0000      1000
          bn     1.0000    0.9480    0.9733      1000
          de     0.9589    0.9810    0.9698      1000
          el     0.9990    0.9910    0.9950      1000
          en     0.8271    0.9900    0.9012      1000
          es     0.9959    0.9770    0.9864      1000
          fr     0.9679    0.9940    0.9808      1000
          hi     0.9980    0.9820    0.9899      1000
          it     0.9896    0.9510    0.9699      1000
          ja     1.0000    0.9900    0.9950      1000
          nl     0.9949    0.9800    0.9874      1000
          pi     0.9987